# KTTS XTTS v1 listening evaluation
Creates review-only Kashmiri WAV samples from the completed XTTS checkpoint. No deployment occurs.


In [ ]:
!nvidia-smi
!git clone -q --depth 1 https://github.com/sameerahmad86232/cipher.git /kaggle/working/cipher
!pip install -q --no-cache-dir 'coqui-tts==0.28.0' soundfile
!python -c "from TTS.tts.models.xtts import Xtts; print('XTTS inference ready')"


In [ ]:
from pathlib import Path
import torch
roots=list(Path('/kaggle/input').rglob('best_model_12684.pth'))
assert len(roots)==1, f'Expected one best checkpoint, found {roots}'
source_checkpoint=roots[0]
config_candidates=list(source_checkpoint.parent.glob('config.json'))
vocab_candidates=list(Path('/kaggle/input').rglob('vocab.json'))
reference_candidates=list(Path('/kaggle/input').rglob('ktts_*.wav'))
assert config_candidates and vocab_candidates and reference_candidates
config_path=config_candidates[0]; vocab_path=vocab_candidates[0]; reference=reference_candidates[0]
print({'checkpoint':str(source_checkpoint),'config':str(config_path),'vocab':str(vocab_path),'reference':str(reference)})
checkpoint=torch.load(source_checkpoint,map_location='cpu',weights_only=False)
checkpoint.pop('optimizer',None); checkpoint.pop('scaler',None)
if isinstance(checkpoint.get('model'),dict):
    for key in [key for key in checkpoint['model'] if 'dvae' in key]: del checkpoint['model'][key]
compact=Path('/kaggle/working/ktts-xtts-v1-inference.pth')
torch.save(checkpoint,compact); del checkpoint
print('Compact inference checkpoint bytes:',compact.stat().st_size)


In [ ]:
import json, sys, zipfile
import soundfile as sf
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts
sys.path.insert(0,'/kaggle/working/cipher/scripts')
from kashmiri_speech_frontend import ascii_g2p
config=XttsConfig(); config.load_json(str(config_path))
model=Xtts.init_from_config(config)
model.load_checkpoint(config,checkpoint_path=str(compact),vocab_path=str(vocab_path),use_deepspeed=False)
model.cuda(); model.eval()
sentences=[
 ('01-greeting','اَلسَلامُ عَلَیکُم، تُہۍ کِتھ پٲٹھۍ چھِو؟'),
 ('02-introduction','مےٚ ناو چھُ سمیر، بٔہ چھُس کویل پُلوامہٕ پٮ۪ٹھ۔'),
 ('03-language','کٲشُر زَبان چھِ واریاہ خوٗبصوٗرَت۔'),
 ('04-long','اَسہِ گَژھِ کٲشُر زَبان صاف لِکھُن، پَرُن تہٕ بولُن۔'),
]
out=Path('/kaggle/working/ktts-xtts-v1-samples'); out.mkdir(exist_ok=True)
manifest=[]
for name,text in sentences:
    phones=ascii_g2p(text)
    result=model.synthesize(phones,config,speaker_wav=str(reference),language='ar',temperature=0.65,length_penalty=1.0,repetition_penalty=8.0,top_k=50,top_p=0.85)
    path=out/f'{name}.wav'; sf.write(path,result['wav'],24000,subtype='PCM_16')
    manifest.append({'file':path.name,'kashmiri':text,'modelInput':phones})
(out/'manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
archive=Path('/kaggle/working/ktts-xtts-v1-listening-samples.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for path in out.iterdir(): z.write(path,path.name)
compact.unlink()
print('Review artifact:',archive,archive.stat().st_size,'bytes')
